# ImageNet Classification for the MLA100 NPU chip


## Load Dataset

In [1]:
!cp /content/drive/MyDrive/NPULab/spring-2026-data/imagenet_train20.txt imagenet_train20.txt
!cp /content/drive/MyDrive/NPULab/spring-2026-data/imagenet_val20.txt imagenet_val20.txt

# Text contains:
# file class_number
# file2 class_number2
# ...

In [2]:
!cp /content/drive/MyDrive/NPULab/spring-2026-data/imagenet_train20.zip imagenet_train20.zip

In [ ]:
!unzip -q imagenet_train20.zip
#   inflating: imagenet_train20a/n04346328/n04346328_2302.JPEG
#  inflating: imagenet_train20a/n04346328/n04346328_2842.JPEG

## Install Requirements

In [ ]:
!pip -q install onnxscript

In [5]:
# Import required libraries
import torch
import torch.nn as nn
import torch.optim as optim
import torch.onnx
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import os
import onnxscript

## Prepare Dataset Loader

In [17]:
IMAGE_ROOT = 'imagenet_train20a'
TRAIN_LIST = 'imagenet_train20.txt'
BATCH_SIZE = 32
NUM_CLASSES = 20
INPUT_SHAPE = (240, 240)
NUM_EPOCHS = 5
LEARNING_RATE = 0.01

In [16]:
class ImageNet20Dataset(Dataset):
    def __init__(self, txt_file, root_dir, transform=None):
        self.img_labels = []
        self.root_dir = root_dir
        self.transform = transform

        with open(txt_file, 'r') as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) >= 2:
                    self.img_labels.append((parts[0], int(parts[1])))

    def __len__(self):
        return len(self.img_labels)

    def __getitem__(self, idx):
        filename, label = self.img_labels[idx]

        folder_name = filename.split('_')[0]

        full_path = os.path.join(self.root_dir, folder_name, filename)

        try:
            image = Image.open(full_path).convert("RGB")
        except Exception as e:
            print(f"Error loading {full_path}: {e}")
            image = Image.new('RGB', INPUT_SHAPE)

        if self.transform:
            image = self.transform(image)

        return image, label

## Prepare SimpleDNN

In [8]:
class SimpleDNN(nn.Module):
    def __init__(self):
        super(SimpleDNN, self).__init__()

        # We use Strided Convolutions (stride=2) instead of MaxPool
        # This reduces dimensions: 240 -> 120 -> 60 -> 30
        self.features = nn.Sequential(
            # Layer 1: 240x240 -> 120x120
            nn.Conv2d(3, 16, kernel_size=(3,3), stride=(2,2), padding=(1,1)),
            nn.ReLU(),

            # Layer 2: 120x120 -> 60x60
            nn.Conv2d(16, 32, kernel_size=(3,3), stride=(2,2), padding=(1,1)),
            nn.ReLU(),

            # Layer 3: 60x60 -> 30x30
            nn.Conv2d(32, 64, kernel_size=(3,3), stride=(2,2), padding=(1,1)),
            nn.ReLU()
        )

        # Calculate Flatten size: 64 channels * 30 width * 30 height
        self.flatten_size = 64 * 30 * 30

        # Final Classification Layer
        self.classifier = nn.Linear(self.flatten_size, NUM_CLASSES)

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1) # Flatten
        x = self.classifier(x)
        return x

## Data Augmentations

In [18]:
transform = transforms.Compose([
    transforms.Resize(INPUT_SHAPE),
    transforms.ToTensor(),
])

try:
    train_dataset = ImageNet20Dataset(txt_file=TRAIN_LIST, root_dir=IMAGE_ROOT, transform=transform)
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    print(f"Dataset loaded: {len(train_dataset)} images found.")
except Exception as e:
    print(f"Dataset setup skipped. Error: {e}")
    train_loader = []

Dataset loaded: 6000 images found.


## Train Model

### Intialize model, criterion and optimizer

In [19]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

model = SimpleDNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=LEARNING_RATE)

Using device: cuda


### Start training loop

In [20]:
if len(train_loader) > 0:
    for epoch in range(NUM_EPOCHS):
        model.train() # Set model to training mode
        running_loss = 0.0

        for i, (images, labels) in enumerate(train_loader):
            # Move data to GPU if available
            images, labels = images.to(device), labels.to(device)

            # Zero the parameter gradients
            optimizer.zero_grad()

            # Forward pass
            outputs = model(images)
            loss = criterion(outputs, labels)

            # Backward pass and optimize
            loss.backward()
            optimizer.step()

            running_loss += loss.item()

            # Print every 10 batches
            if (i + 1) % 10 == 0:
                print(f"Epoch [{epoch+1}/{NUM_EPOCHS}], Step [{i+1}/{len(train_loader)}], Loss: {loss.item():.4f}")

        epoch_loss = running_loss / len(train_loader)
        print(f"Epoch [{epoch+1}/{NUM_EPOCHS}] Finished. Average Loss: {epoch_loss:.4f}")
else:
    print("Skipping training loop (Dataset empty or not found).")

Epoch [1/5], Step [10/188], Loss: 3.0001
Epoch [1/5], Step [20/188], Loss: 3.0068
Epoch [1/5], Step [30/188], Loss: 3.0135
Epoch [1/5], Step [40/188], Loss: 2.9793
Epoch [1/5], Step [50/188], Loss: 2.9482
Epoch [1/5], Step [60/188], Loss: 2.9459
Epoch [1/5], Step [70/188], Loss: 2.9773
Epoch [1/5], Step [80/188], Loss: 2.9711
Epoch [1/5], Step [90/188], Loss: 2.9579
Epoch [1/5], Step [100/188], Loss: 2.9671
Epoch [1/5], Step [110/188], Loss: 2.9190
Epoch [1/5], Step [120/188], Loss: 2.8621
Epoch [1/5], Step [130/188], Loss: 2.9029
Epoch [1/5], Step [140/188], Loss: 2.8541
Epoch [1/5], Step [150/188], Loss: 2.8163
Epoch [1/5], Step [160/188], Loss: 2.8552
Epoch [1/5], Step [170/188], Loss: 2.8260
Epoch [1/5], Step [180/188], Loss: 2.8554
Epoch [1/5] Finished. Average Loss: 2.9347
Epoch [2/5], Step [10/188], Loss: 2.5980
Epoch [2/5], Step [20/188], Loss: 2.7516
Epoch [2/5], Step [30/188], Loss: 2.7223
Epoch [2/5], Step [40/188], Loss: 2.6452
Epoch [2/5], Step [50/188], Loss: 2.8821
Epoch

## Export Model to ONNX

In [21]:
model.eval()

# Move dummy input to same device as model
dummy_input = torch.randn(1, 3, 240, 240).to(device)
onnx_filename = "simple_imagenet20_model.onnx"

torch.onnx.export(
    model,
    dummy_input,
    onnx_filename,
    verbose=False,
    input_names=['input'],
    output_names=['output'],
    dynamic_axes={'input': {0: 'batch_size'}, 'output': {0: 'batch_size'}}
)

print(f"Model successfully exported to {onnx_filename}")

/tmp/ipython-input-2262829125.py:7: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(
W0206 03:26:28.265000 1052 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0, sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
W0206 03:26:28.266000 1052 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'boxes' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0, sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
W0206 03:26:28.268000 1052 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0

Applied 1 of general pattern rewrite rules.
Model successfully exported to simple_imagenet20_model.onnx


## Test Compilation

Using [https://github.com/bencejdanko/compiler-evaluation-server](https://github.com/bencejdanko/compiler-evaluation-server) as a compilation server.

In [22]:
%%bash
NGROK_URL="https://3511-73-162-139-161.ngrok-free.app"

echo "$NGROK_URL/compile"

curl -X POST "$NGROK_URL/compile" \
  -F "file=@simple_imagenet20_model.onnx" \
  -F "output_name=custom_output.mxq"

https://3511-73-162-139-161.ngrok-free.app/compile
{"status":"success","stdout":"\n[⣾] Parsing deep learning model....\u001b[K\n[⣽] Parsing deep learning model.....\u001b[K\n[⣻] Parsing deep learning model......\u001b[K\n[⢿] Parsing deep learning model...\u001b[K\n[⡿] Parsing deep learning model....\u001b[K\n[⣟] Parsing deep learning model.....\u001b[K\n\n\n\n\n✔ Parsing deep learning model... Done!          \n[2026-02-06 12:26:47.791] Setting Quantizer Config...\nOPTQ config is found\noptq apply false\nminOutputDiff config is found\ntransformerQuant is found\nbitAllocation config is found\nquantLayer config is found\nllm config is found\nreduceGPU config is found\n[2026-02-06 12:26:47.797] Performing Quantization...\n[2026-02-06 12:26:47.821] complete high-level model optimizing\n[2026-02-06 12:26:47.839] random input name: input_0, input shape: [1, 240, 240, 3], data type: float32\n\n================ Input Information ===============\nInput Name        = input_0\nInput Shape       = 

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 25447  100 12004  100 13443    703    787  0:00:17  0:00:17 --:--:--  3160
